# Split Data (DT+/DT-) by distance from reference

Questo notebook esegue:
1. lettura di DN, DT-, DT+;
2. selezione di DT- e DT+ con distanza dalla reference in [25, 30];
3. split train/test al 15% per ciascun dataset DT- e DT+;
4. salvataggio dei FASTA richiesti;
5. filtro dei test split in [25, 30] e salvataggio in due file aggiuntivi;
6. split dei soli record con "vae" nell'header (DT- e DT+) in dentro/fuori [25,30] in una cartella separata.

In [ ]:
from pathlib import Path
import numpy as np

from adabmDCA.fasta import get_tokens, import_from_fasta


def hamming_distance_to_reference(seqs: np.ndarray, reference: np.ndarray) -> np.ndarray:
    if seqs.ndim != 2 or reference.ndim != 1:
        raise ValueError("`seqs` deve essere 2D e `reference` 1D")
    if seqs.shape[1] != reference.shape[0]:
        raise ValueError("Lunghezza sequenze e reference non compatibile")
    return (seqs != reference).sum(axis=1).astype(np.int32)


def write_fasta(headers: np.ndarray, seqs: np.ndarray, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # If sequences are integer-encoded, decode using current RNA tokens.
    tok = globals().get("tokens", None)

    with out_path.open("w", encoding="utf-8") as f:
        for h, s in zip(headers, seqs):
            h_str = str(h)
            if not h_str.startswith(">"):
                h_str = ">" + h_str

            s_arr = np.asarray(s)
            if np.issubdtype(s_arr.dtype, np.integer):
                if tok is None:
                    raise ValueError("Token RNA non disponibili per decodificare sequenze intere")
                seq_str = "".join(tok[int(x)] for x in s_arr.tolist())
            else:
                seq_str = "".join(s_arr.astype(str).tolist())

            f.write(h_str + "\n")
            f.write(seq_str + "\n")


def split_train_test(headers: np.ndarray, seqs: np.ndarray, test_frac: float = 0.15, seed: int = 42):
    if not (0.0 < test_frac < 1.0):
        raise ValueError("`test_frac` deve essere in (0, 1)")

    n = seqs.shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)

    n_test = int(np.floor(test_frac * n))
    n_test = max(1, n_test)

    test_idx = idx[:n_test]
    train_idx = idx[n_test:]

    return (
        headers[train_idx], seqs[train_idx],
        headers[test_idx], seqs[test_idx],
    )

In [ ]:
# Lettura dati
tokens = get_tokens("rna")

headers_DN, data_DN = import_from_fasta(
    "Group_I_intron-2/DN&DT/DN.fasta",
    tokens=tokens,
    filter_sequences=True,
)
headers_DTm, data_DTm = import_from_fasta(
    "Group_I_intron-2/DN&DT/DT-.fasta",
    tokens=tokens,
    filter_sequences=True,
)
headers_DTp, data_DTp = import_from_fasta(
    "Group_I_intron-2/DN&DT/DT+.fasta",
    tokens=tokens,
    filter_sequences=True,
)

print("Shape DN :", data_DN.shape)
print("Shape DT-:", data_DTm.shape)
print("Shape DT+:", data_DTp.shape)

reference_seq = data_DN[0]

dist_tm = hamming_distance_to_reference(data_DTm, reference_seq)
dist_tp = hamming_distance_to_reference(data_DTp, reference_seq)

# Intervallo richiesto [25, 30]
low, high = 25, 30
mask_tm_25_30 = (dist_tm >= low) & (dist_tm <= high)
mask_tp_25_30 = (dist_tp >= low) & (dist_tp <= high)

headers_tm_25_30 = headers_DTm[mask_tm_25_30]
data_tm_25_30 = data_DTm[mask_tm_25_30]
headers_tp_25_30 = headers_DTp[mask_tp_25_30]
data_tp_25_30 = data_DTp[mask_tp_25_30]

print(f"DT- in [25,30]: {data_tm_25_30.shape[0]}")
print(f"DT+ in [25,30]: {data_tp_25_30.shape[0]}")

In [ ]:
# Salvataggio file richiesti
out_dir = Path("Group_I_intron-2/DN&DT/split_data")
out_dir.mkdir(parents=True, exist_ok=True)


def split_inside_outside(headers: np.ndarray, seqs: np.ndarray, reference: np.ndarray, low: int, high: int):
    dist = hamming_distance_to_reference(seqs, reference)
    mask_in = (dist >= low) & (dist <= high)
    mask_out = ~mask_in
    return (
        headers[mask_in], seqs[mask_in],
        headers[mask_out], seqs[mask_out],
    )


# -----------------------------------------------------------------------------
# (A) Dataset globale: DT- e DT+ dentro/fuori [25,30]
# -----------------------------------------------------------------------------
path_tm_in_global = out_dir / "DTm_dist_25_30.fasta"
path_tm_out_global = out_dir / "DTm_dist_outside_25_30.fasta"
path_tp_in_global = out_dir / "DTp_dist_25_30.fasta"
path_tp_out_global = out_dir / "DTp_dist_outside_25_30.fasta"

write_fasta(headers_tm_25_30, data_tm_25_30, path_tm_in_global)
write_fasta(headers_tp_25_30, data_tp_25_30, path_tp_in_global)

mask_tm_out_25_30 = ~mask_tm_25_30
mask_tp_out_25_30 = ~mask_tp_25_30
headers_tm_out_25_30 = headers_DTm[mask_tm_out_25_30]
data_tm_out_25_30 = data_DTm[mask_tm_out_25_30]
headers_tp_out_25_30 = headers_DTp[mask_tp_out_25_30]
data_tp_out_25_30 = data_DTp[mask_tp_out_25_30]

write_fasta(headers_tm_out_25_30, data_tm_out_25_30, path_tm_out_global)
write_fasta(headers_tp_out_25_30, data_tp_out_25_30, path_tp_out_global)

# -----------------------------------------------------------------------------
# (B) Split train/test al 15% separato per DT- e DT+
# -----------------------------------------------------------------------------
seed = 42
tm_h_tr, tm_x_tr, tm_h_te, tm_x_te = split_train_test(headers_DTm, data_DTm, test_frac=0.15, seed=seed)
tp_h_tr, tp_x_tr, tp_h_te, tp_x_te = split_train_test(headers_DTp, data_DTp, test_frac=0.15, seed=seed)

# Salvo train/test separati per classe
path_tm_train = out_dir / "DTm_train_split_85.fasta"
path_tm_test = out_dir / "DTm_test_split_15.fasta"
path_tp_train = out_dir / "DTp_train_split_85.fasta"
path_tp_test = out_dir / "DTp_test_split_15.fasta"

write_fasta(tm_h_tr, tm_x_tr, path_tm_train)
write_fasta(tm_h_te, tm_x_te, path_tm_test)
write_fasta(tp_h_tr, tp_x_tr, path_tp_train)
write_fasta(tp_h_te, tp_x_te, path_tp_test)

# -----------------------------------------------------------------------------
# (C) Train/Test separati per classe e per intervallo dentro/fuori [25,30]
# -----------------------------------------------------------------------------
# DT- train
(tm_h_tr_in, tm_x_tr_in, tm_h_tr_out, tm_x_tr_out) = split_inside_outside(
    tm_h_tr, tm_x_tr, reference_seq, low, high
)
# DT- test
(tm_h_te_in, tm_x_te_in, tm_h_te_out, tm_x_te_out) = split_inside_outside(
    tm_h_te, tm_x_te, reference_seq, low, high
)
# DT+ train
(tp_h_tr_in, tp_x_tr_in, tp_h_tr_out, tp_x_tr_out) = split_inside_outside(
    tp_h_tr, tp_x_tr, reference_seq, low, high
)
# DT+ test
(tp_h_te_in, tp_x_te_in, tp_h_te_out, tp_x_te_out) = split_inside_outside(
    tp_h_te, tp_x_te, reference_seq, low, high
)

# Percorsi DT-
path_tm_train_in = out_dir / "DTm_train_split_85_dist_25_30.fasta"
path_tm_train_out = out_dir / "DTm_train_split_85_dist_outside_25_30.fasta"
path_tm_test_in = out_dir / "DTm_test_split_15_dist_25_30.fasta"
path_tm_test_out = out_dir / "DTm_test_split_15_dist_outside_25_30.fasta"

# Percorsi DT+
path_tp_train_in = out_dir / "DTp_train_split_85_dist_25_30.fasta"
path_tp_train_out = out_dir / "DTp_train_split_85_dist_outside_25_30.fasta"
path_tp_test_in = out_dir / "DTp_test_split_15_dist_25_30.fasta"
path_tp_test_out = out_dir / "DTp_test_split_15_dist_outside_25_30.fasta"

# Salvataggio DT-
write_fasta(tm_h_tr_in, tm_x_tr_in, path_tm_train_in)
write_fasta(tm_h_tr_out, tm_x_tr_out, path_tm_train_out)
write_fasta(tm_h_te_in, tm_x_te_in, path_tm_test_in)
write_fasta(tm_h_te_out, tm_x_te_out, path_tm_test_out)

# Salvataggio DT+
write_fasta(tp_h_tr_in, tp_x_tr_in, path_tp_train_in)
write_fasta(tp_h_tr_out, tp_x_tr_out, path_tp_train_out)
write_fasta(tp_h_te_in, tp_x_te_in, path_tp_test_in)
write_fasta(tp_h_te_out, tp_x_te_out, path_tp_test_out)

print("\nFile creati (globali dentro/fuori):")
print(path_tm_in_global)
print(path_tm_out_global)
print(path_tp_in_global)
print(path_tp_out_global)

print("\nFile creati (split train/test per classe):")
print(path_tm_train)
print(path_tm_test)
print(path_tp_train)
print(path_tp_test)

print("\nFile creati (split per classe dentro/fuori intervallo):")
print(path_tm_train_in)
print(path_tm_train_out)
print(path_tm_test_in)
print(path_tm_test_out)
print(path_tp_train_in)
print(path_tp_train_out)
print(path_tp_test_in)
print(path_tp_test_out)

print("\nCardinalita globali:")
print(f"DT- in [25,30]: {data_tm_25_30.shape[0]} | fuori: {data_tm_out_25_30.shape[0]}")
print(f"DT+ in [25,30]: {data_tp_25_30.shape[0]} | fuori: {data_tp_out_25_30.shape[0]}")

print("\nCardinalita split per classe:")
print(f"DT- train/test: {tm_x_tr.shape[0]} / {tm_x_te.shape[0]}")
print(f"DT+ train/test: {tp_x_tr.shape[0]} / {tp_x_te.shape[0]}")

print("\nCardinalita split DT- (inside/outside):")
print(f"train: {tm_x_tr_in.shape[0]} / {tm_x_tr_out.shape[0]}")
print(f"test : {tm_x_te_in.shape[0]} / {tm_x_te_out.shape[0]}")

print("\nCardinalita split DT+ (inside/outside):")
print(f"train: {tp_x_tr_in.shape[0]} / {tp_x_tr_out.shape[0]}")
print(f"test : {tp_x_te_in.shape[0]} / {tp_x_te_out.shape[0]}")

In [ ]:
# -----------------------------------------------------------------------------
# (D) Split VAE in cartella separata: distanza dalla reference dentro/fuori [25,30]
# -----------------------------------------------------------------------------
vae_dir = out_dir / "vae_split_25_30"
vae_dir.mkdir(parents=True, exist_ok=True)


def header_contains_vae(headers: np.ndarray) -> np.ndarray:
    return np.array(["vae" in str(h).lower() for h in headers], dtype=bool)


mask_tm_vae = header_contains_vae(headers_DTm)
mask_tp_vae = header_contains_vae(headers_DTp)

headers_tm_vae = headers_DTm[mask_tm_vae]
data_tm_vae = data_DTm[mask_tm_vae]
headers_tp_vae = headers_DTp[mask_tp_vae]
data_tp_vae = data_DTp[mask_tp_vae]

(tm_h_vae_in, tm_x_vae_in, tm_h_vae_out, tm_x_vae_out) = split_inside_outside(
    headers_tm_vae, data_tm_vae, reference_seq, low, high
)
(tp_h_vae_in, tp_x_vae_in, tp_h_vae_out, tp_x_vae_out) = split_inside_outside(
    headers_tp_vae, data_tp_vae, reference_seq, low, high
)

path_tm_vae_in = vae_dir / "DTm_vae_dist_25_30.fasta"
path_tm_vae_out = vae_dir / "DTm_vae_dist_outside_25_30.fasta"
path_tp_vae_in = vae_dir / "DTp_vae_dist_25_30.fasta"
path_tp_vae_out = vae_dir / "DTp_vae_dist_outside_25_30.fasta"

write_fasta(tm_h_vae_in, tm_x_vae_in, path_tm_vae_in)
write_fasta(tm_h_vae_out, tm_x_vae_out, path_tm_vae_out)
write_fasta(tp_h_vae_in, tp_x_vae_in, path_tp_vae_in)
write_fasta(tp_h_vae_out, tp_x_vae_out, path_tp_vae_out)

print("\nSplit VAE creato in:", vae_dir)
print("File VAE DT-:", path_tm_vae_in, "|", path_tm_vae_out)
print("File VAE DT+:", path_tp_vae_in, "|", path_tp_vae_out)
print("Cardinalita VAE DT- in/out:", tm_x_vae_in.shape[0], "/", tm_x_vae_out.shape[0])
print("Cardinalita VAE DT+ in/out:", tp_x_vae_in.shape[0], "/", tp_x_vae_out.shape[0])